# Build the Kaggle submission files

Combines the per-task outputs in `../outputs/` into the two files selected for the final ranking:

| File | Kaggle public score | Task 3 | Task 4 | Task 5 |
|---|---|---|---|---|
| **V57a** `submission_v57a.csv` | **0.84167** | blend 0.55 C + 0.25 A + 0.20 B | six new-recipe DINOv2-L + DINOv2-B + others, EM class prior | 1,857 flags |
| **V53c** `submission_v53c.csv` | 0.84113 | 0.5 A + 0.5 B (earlier blend) | four new-recipe DINOv2-L + DINOv2-B (80% data) + others, EM class prior | 1,855 flags (no tail rows) |

Tasks 1 and 2 are identical in both. Row `i` of every task's `test.csv` belongs to row `i` of `sample_submission.csv` (ids `ANS00001`...). If the files that were uploaded to Kaggle are available, set `REF_V57A` / `REF_V53C` to compare.

In [1]:
import os, re, sys, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ---- configuration -------------------------------------------------------------------------------
# DATA = folder of the Kaggle download that holds 01_binary_classification/ ... 05_anomaly_detection/ and sample_submission.csv
DATA = Path(os.environ.get("CPE342_DATA", "")).expanduser()
assert (DATA / "01_binary_classification").exists(), "Set the environment variable CPE342_DATA to the Kaggle download folder (see ../README.md)"
OUT = Path(os.environ.get("CPE342_OUT", "../outputs")); OUT.mkdir(parents=True, exist_ok=True)   # shared by all notebooks
SEED = 342                                   # every random split / model seed in the project
NJOBS = int(os.environ.get("NJOBS", 16))     # LightGBM threads

In [2]:
sample = pd.read_csv(DATA / "sample_submission.csv")
two_raw, l1_id = np.load(OUT / "task3_test_two_raw.npy"), np.load(OUT / "task3_test_l1_identity.npy")
parts = {
    "v57a": dict(task3=np.load(OUT / "task3_pred.npy"), task4="task4_pred_v57a.npy", task5=np.load(OUT / "task5_pred.npy")),
    "v53c": dict(task3=np.clip(0.5 * two_raw + 0.5 * l1_id, 0, 300000), task4="task4_pred_v53c.npy", task5=np.load(OUT / "task5_pred_without_tail.npy")),
}
for name, p in parts.items():
    if not (OUT / p["task4"]).exists(): print(f"{name}: Task 4 output missing, skipped"); continue
    sub = pd.DataFrame({"id": sample.id})
    sub["task1"] = np.load(OUT / "task1_pred.npy").astype(int); sub["task2"] = np.load(OUT / "task2_pred.npy").astype(int)
    sub["task3"] = p["task3"]; sub["task4"] = np.load(OUT / p["task4"]).astype(int); sub["task5"] = p["task5"].astype(int)
    # format checks
    assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample) and not sub.isna().any().any()
    assert set(sub.task1) <= {0, 1} and set(sub.task2) <= {0, 1, 2, 3} and set(sub.task4) <= {0, 1, 2, 3, 4} and set(sub.task5) <= {0, 1} and sub.task3.between(0, 300000).all()
    sub.to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"{name}: {sub.shape} | task1 flagged {sub.task1.mean():.1%} | task2 shares {np.round(np.bincount(sub.task2) / len(sub), 3).tolist()} | task3 mean {sub.task3.mean():.0f} | "
          f"task4 shares {np.round(np.bincount(sub.task4) / len(sub), 3).tolist()} | task5 flags {int(sub.task5.sum())}")
    ref = os.environ.get(f"REF_{name.upper()}")
    if ref:
        r = pd.read_csv(ref); assert r.id.tolist() == sub.id.tolist()
        print("   vs the file uploaded to Kaggle:", {c: (f"{(r[c] == sub[c]).mean():.4%} equal" if c != "task3" else f"mean abs diff {np.abs(r[c] - sub[c]).mean():.3f}") for c in sub.columns[1:]})

v57a: (25889, 6) | task1 flagged 60.2% | task2 shares [0.416, 0.249, 0.195, 0.14] | task3 mean 10435 | task4 shares [0.135, 0.289, 0.139, 0.249, 0.187] | task5 flags 1857
   vs the file uploaded to Kaggle: {'task1': '100.0000% equal', 'task2': '100.0000% equal', 'task3': 'mean abs diff 0.000', 'task4': '100.0000% equal', 'task5': '100.0000% equal'}
v53c: (25889, 6) | task1 flagged 60.2% | task2 shares [0.416, 0.249, 0.195, 0.14] | task3 mean 10411 | task4 shares [0.135, 0.289, 0.139, 0.249, 0.188] | task5 flags 1855
   vs the file uploaded to Kaggle: {'task1': '100.0000% equal', 'task2': '100.0000% equal', 'task3': 'mean abs diff 0.000', 'task4': '100.0000% equal', 'task5': '100.0000% equal'}
